# 轮动策略教程

本教程介绍如何用 open-xquant 构建一个**动量轮动策略**——在多个资产中，按风险调整后的动量排名，定期选择表现最优的资产并分配权重。

我们将使用三只跨资产类别 ETF 作为投资标的：

| 代码 | 名称 | 资产类别 |
|------|------|----------|
| 513100.SS | 纳指100ETF | 美股科技 |
| 510300.SS | 沪深300ETF | A股大盘 |
| 518880.SS | 黄金ETF | 贵金属 |

### 与均线策略的区别

在 `engine_module` 教程中，我们学习了 SMA 均线交叉策略——**单标的、固定仓位、信号驱动**。轮动策略则是一种完全不同的范式：

| 维度 | 均线交叉策略 | 轮动策略 |
|------|-------------|----------|
| 标的数 | 单标的 | **多标的截面选择** |
| 信号含义 | 买入/不买入（布尔） | **目标权重**（0.0~1.0） |
| 仓位管理 | 固定股数 or 全仓 | **按权重动态调仓** |
| 权重分配 | 无 | **PortfolioOptimizer** |

### 新组件一览

本教程将用到以下组件：

- **Momentum** — N 日动量指标（平均日对数收益率）
- **RollingVolatility** — N 日滚动波动率
- **Ratio** — 两列之比（用于计算风险调整动量）
- **Threshold** — per-symbol 布尔信号（用于携带 required_indicators）
- **TopNRankingOptimizer** — PortfolioOptimizer，按评分排名选 Top N 并归一化权重

## 1. 准备数据

In [ ]:
from oxq.data import YFinanceDownloader

dl = YFinanceDownloader()
for s in ["513100.SS", "510300.SS", "518880.SS"]:
    path = dl.download(s, "2024-11-15", "2026-02-28")
    print(f"{s}: {path}")

In [ ]:
from oxq.data import LocalMarketDataProvider

market = LocalMarketDataProvider()
for s in ["513100.SS", "510300.SS", "518880.SS"]:
    df = market.get_bars(s, "2024-11-15", "2026-02-28")
    print(f"{s}: {len(df)} bars, {df.index[0].date()} ~ {df.index[-1].date()}")

---
## 2. Indicator 层 — Momentum, RollingVolatility, Ratio

轮动策略的核心思路：**买动量强、波动低的资产**。我们需要三个指标：

1. **Momentum(20)** — 过去 20 天的平均日对数收益率，衡量趋势强度
2. **RollingVolatility(20)** — 过去 20 天的滚动波动率，衡量风险
3. **Ratio(mom/vol)** — 动量除以波动率 = 风险调整动量（RAM）

RAM 越高，说明该资产趋势越强且波动越低——是轮动策略最想持有的。

In [ ]:
from oxq.indicators import Momentum, RollingVolatility, Ratio

# 以纳指100ETF为例，演示三个指标的计算过程
df = market.get_bars("513100.SS", "2024-11-15", "2026-02-28").copy()

df["mom"] = Momentum().compute(df, column="close", period=20)
df["vol"] = RollingVolatility().compute(df, column="close", period=20)
df["ram"] = Ratio().compute(df, col_a="mom", col_b="vol")

print("纳指100ETF 指标计算结果（最后 10 行）：")
print(df[["close", "mom", "vol", "ram"]].tail(10).to_string(float_format="%.4f"))

可以看到：
- `mom` — 前 20 行为 NaN（窗口不足），之后反映近 20 日的平均日收益率
- `vol` — 前 ~21 行为 NaN（log_returns.diff() 再 rolling），反映近 20 日的波动水平
- `ram` — mom/vol，值越大说明"趋势强且稳定"，是排名的依据

**这就是为什么策略开头有一段空仓期** — 指标需要约 21 个交易日的预热数据。

我们同时计算三只 ETF 的 RAM 并对比：

In [ ]:
import pandas as pd

SYMBOLS = {"513100.SS": "纳指100ETF", "510300.SS": "沪深300ETF", "518880.SS": "黄金ETF"}

ram_df = pd.DataFrame()
for sym, name in SYMBOLS.items():
    d = market.get_bars(sym, "2024-11-15", "2026-02-28").copy()
    d["mom"] = Momentum().compute(d, column="close", period=20)
    d["vol"] = RollingVolatility().compute(d, column="close", period=20)
    d["ram"] = Ratio().compute(d, col_a="mom", col_b="vol")
    ram_df[name] = d["ram"]

# 显示 2025 年 1 月的 RAM 截面对比
print("2025年1月 风险调整动量（RAM）截面对比：")
jan = ram_df.loc["2025-01-01":"2025-01-31"]
print(jan.to_string(float_format="%.4f"))

每天三只 ETF 都有各自的 RAM 值。**哪只 RAM 最高，就应该分配更多权重**——这正是 TopNRankingOptimizer 要做的事。

---
## 3. PortfolioOptimizer 层 — TopNRankingOptimizer

在新架构中，截面排名和权重分配由 **TopNRankingOptimizer**（PortfolioOptimizer）负责，而非 Signal。Signal 层只负责 per-symbol 的布尔/数值计算。

| 维度 | Crossover (Signal) | TopNRankingOptimizer (Portfolio) |
|------|-----------|-------------|
| 输出类型 | 布尔值（True/False） | **权重**（0.0~1.0） |
| 视角 | 单 symbol 时序 | **多 symbol 截面** |
| 含义 | "此刻是否应该买入" | "此刻应该分配多少仓位" |

TopNRankingOptimizer 的处理流程（每个 bar）：

```
1. 读取每个 symbol 的评分列（如 ram）
2. 过滤 NaN 和负值（动量为负 = 下跌趋势，不选）
3. 按评分降序排名，取 Top N
4. 归一化为权重（评分越高，权重越大，总和 = 1.0）
5. 权重上限裁剪（如单只不超过 60%，超出部分归现金）
```

In [ ]:
from oxq.portfolio.optimizers import TopNRankingOptimizer

# 准备 indicators_data: dict[symbol, DataFrame]（每个 DataFrame 包含 ram 列）
indicators_data = {}
signals_data = {}
for sym in SYMBOLS:
    d = market.get_bars(sym, "2024-11-15", "2026-02-28").copy()
    d["mom"] = Momentum().compute(d, column="close", period=20)
    d["vol"] = RollingVolatility().compute(d, column="close", period=20)
    d["ram"] = Ratio().compute(d, col_a="mom", col_b="vol")
    indicators_data[sym] = d
    signals_data[sym] = d

# TopNRankingOptimizer: 选 Top 2，单只上限 60%
optimizer = TopNRankingOptimizer(score_col="ram", n=2, max_weight=0.6)
weights = optimizer.optimize(signals_data, indicators_data)

print("TopNRankingOptimizer 目标权重（最后一个 bar）：")
for sym, w in weights.items():
    name = SYMBOLS.get(sym, sym)
    print(f"  {name}: {w:.4f}")

TopNRankingOptimizer 在每个 bar 执行截面排名：
- 只有 **最多 2 只** ETF 获得非零权重（n=2）
- 单只权重不超过 **0.6**（max_weight=0.6），超出部分不会分给其他标的，而是留作现金
- 动量为负的标的权重为 0（被 filter_negative 过滤）

### 权重上限的设计意图

为什么超过 max_weight 的部分归现金而不是重新分配？

假设某天 A 的 RAM 远高于 B，归一化后 A=0.9, B=0.1。如果 max_weight=0.6：

- **归现金**（本策略采用）：A=0.6, B=0.1, Cash=0.3 — 自然降低了总仓位
- **重新分配**：A=0.6, B=0.4 — 人为放大了弱势标的的权重

前者更保守，在信号集中度高的时候自动降杠杆。

---
## 4. PortfolioOptimizer — 权重分配

在新架构中，仓位管理由 **PortfolioOptimizer** 负责，而不是 RebalanceRule。PortfolioOptimizer 根据 indicators 数据决定最终的仓位分配。

open-xquant 提供四种内置优化器：

| 优化器 | 策略 |
|--------|------|
| `EqualWeightOptimizer` | 等权分配 |
| `RiskParityOptimizer` | 按波动率反比分配 |
| `KellyOptimizer` | Kelly 公式分配 |
| `TopNRankingOptimizer` | 按评分排名选 Top N |

对于轮动策略，我们使用 `TopNRankingOptimizer` 作为 PortfolioOptimizer，它直接读取 indicators 中的评分列（如 ram）进行截面排名和权重分配。

In [ ]:
from oxq.portfolio import EqualWeightOptimizer, RiskParityOptimizer

# 演示 EqualWeightOptimizer
eq_opt = EqualWeightOptimizer()

# 准备模拟的 signals 和 indicators 数据
demo_signals = {
    "513100.SS": pd.DataFrame({"tw": [0.5]}, index=[pd.Timestamp("2025-01-06")]),
    "518880.SS": pd.DataFrame({"tw": [0.3]}, index=[pd.Timestamp("2025-01-06")]),
}
demo_indicators = {
    "513100.SS": pd.DataFrame({"ram": [0.8]}, index=[pd.Timestamp("2025-01-06")]),
    "518880.SS": pd.DataFrame({"ram": [0.5]}, index=[pd.Timestamp("2025-01-06")]),
}

weights = eq_opt.optimize(demo_signals, demo_indicators)
print(f"EqualWeightOptimizer 输出: {weights}")
print(f"  每只标的权重 = 1/{len(demo_signals)} = {1/len(demo_signals):.4f}")

In [ ]:
# 演示 RiskParityOptimizer — 按波动率反比分配权重
rp_opt = RiskParityOptimizer(volatility_col="vol")

demo_indicators_rp = {
    "513100.SS": pd.DataFrame({"vol": [0.02]}, index=[pd.Timestamp("2025-01-06")]),
    "510300.SS": pd.DataFrame({"vol": [0.03]}, index=[pd.Timestamp("2025-01-06")]),
    "518880.SS": pd.DataFrame({"vol": [0.01]}, index=[pd.Timestamp("2025-01-06")]),
}
demo_signals_rp = {s: pd.DataFrame() for s in demo_indicators_rp}

rp_weights = rp_opt.optimize(demo_signals_rp, demo_indicators_rp)
print("RiskParityOptimizer 输出（波动率反比）:")
for sym, w in rp_weights.items():
    vol = demo_indicators_rp[sym]["vol"].iloc[0] if sym in demo_indicators_rp else 0
    print(f"  {sym}: 权重={w:.4f}  (vol={vol})")

In [ ]:
print("PortfolioOptimizer 小结：")
print("  - EqualWeightOptimizer:  不看信号内容，等权分配所有标的")
print("  - RiskParityOptimizer:   按波动率反比分配，低波动标的获得更多权重")
print("  - KellyOptimizer:        根据胜率和盈亏比计算最优仓位")
print("  - TopNRankingOptimizer:  按评分列排名，选 Top N 并归一化权重")
print()
print("在轮动策略中，TopNRankingOptimizer 直接读取 indicators 中的评分列，")
print("完成截面排名和权重分配。Signal 层只负责 per-symbol 布尔信号。")

### 新旧架构对比

| 维度 | 旧架构 | 新架构 |
|------|--------|--------|
| 指标声明 | Strategy.indicators | Signal.required_indicators |
| 权重分配 | RebalanceRule | **PortfolioOptimizer** |
| 规则传入 | Strategy.rebalance_rules | **Engine.run(rules=[...])** |
| 入场/出场 | EntryRule / ExitRule | **PortfolioOptimizer + Rules** |

In [ ]:
# 新架构的核心变化演示：
# 1. 指标通过 signal.required_indicators 声明（而非 Strategy.indicators）
# 2. PortfolioOptimizer 替代了 RebalanceRule
# 3. 截面排名由 TopNRankingOptimizer 处理，不再是 Signal

from oxq.indicators import Momentum, RollingVolatility, Ratio
from oxq.signals import Threshold

# 创建信号并设置 required_indicators（Threshold 作为 indicator 载体）
active_signal = Threshold()
active_signal.required_indicators = {
    "mom": (Momentum(), {"column": "close", "period": 20}),
    "vol": (RollingVolatility(), {"column": "close", "period": 20}),
    "ram": (Ratio(), {"col_a": "mom", "col_b": "vol"}),
}

print("Signal: Threshold (作为 indicator 载体)")
print(f"  required_indicators: {list(active_signal.required_indicators.keys())}")
print("  Engine 会自动先计算这些指标，再运行信号")
print()
print("Portfolio: TopNRankingOptimizer")
print("  截面排名和权重分配由 PortfolioOptimizer 层处理")

---
## 5. 组装完整策略

现在把所有组件组合为一个完整的 Strategy：

```
Universe    → 513100.SS, 510300.SS, 518880.SS
  ↓
Signal      → active = Threshold(column=ram, threshold=0, relationship=gt)
              └─ required_indicators:
                   mom = Momentum(20)
                   vol = RollingVolatility(20)
                   ram = Ratio(mom / vol)
  ↓
Portfolio   → TopNRankingOptimizer(score_col=ram, n=2, max_weight=0.6)
```

注意新架构的变化：
- **indicators 不再在 Strategy 中声明**——它们通过 Signal 的 `required_indicators` 属性自动收集
- **截面排名由 TopNRankingOptimizer 处理**——不再是 Signal 层的职责
- **Strategy 只有四个核心属性**：name, universe, signals, portfolio

In [ ]:
from oxq.core import Engine, Strategy
from oxq.portfolio.optimizers import TopNRankingOptimizer
from oxq.trade import SimBroker
from oxq.universe import StaticUniverse

# 创建信号并设置 required_indicators
active_signal = Threshold()
active_signal.required_indicators = {
    "mom": (Momentum(), {"column": "close", "period": 20}),
    "vol": (RollingVolatility(), {"column": "close", "period": 20}),
    "ram": (Ratio(), {"col_a": "mom", "col_b": "vol"}),
}

strategy = Strategy(
    name="momentum_rotation",
    universe=StaticUniverse(("513100.SS", "510300.SS", "518880.SS")),
    signals={
        "active": (active_signal, {"column": "ram", "threshold": 0, "relationship": "gt"}),
    },
    portfolio=TopNRankingOptimizer(score_col="ram", n=2, max_weight=0.6),
    hypothesis=(
        "在纳指100ETF、沪深300ETF、黄金ETF中，"
        "按 Momentum(20)/Volatility(20) 风险调整动量排名，"
        "选 Top 2 归一化权重，单只上限 60%，"
        "可获得正超额收益"
    ),
)

print(f"策略名称: {strategy.name}")
print(f"信号: {list(strategy.signals.keys())}")
print(f"信号依赖的指标: {list(active_signal.required_indicators.keys())}")
print(f"Portfolio优化器: {strategy.portfolio.name}")

---
## 6. 运行回测

In [ ]:
broker = SimBroker()
result = Engine().run(
    strategy,
    market=LocalMarketDataProvider(),
    broker=broker,
    start="2024-11-15",
    end="2026-02-28",
    initial_cash=100_000.0,
)

print(f"总收益率:     {result.total_return():.2%}")
print(f"年化收益率:   {result.annualized_return():.2%}")
print(f"年化波动率:   {result.annualized_volatility():.2%}")
print(f"Sharpe Ratio: {result.sharpe_ratio():.2f}")
print(f"Calmar Ratio: {result.calmar_ratio():.2f}")
print(f"Sortino Ratio: {result.sortino_ratio():.2f}")
print(f"最大回撤:     {result.max_drawdown():.2%}")
print(f"交易次数:     {len(result.trades)}")
print(f"期末总资产:   {result.equity_curve[-1][1]:,.0f}")

---
## 7. 查看交易记录

In [ ]:
NAMES = {"513100.SS": "纳指100ETF", "510300.SS": "沪深300ETF", "518880.SS": "黄金ETF"}

print(f"{'日期':<25} {'方向':>4}  {'数量':>6}  {'标的':<12} {'成交价':>10}")
print("-" * 65)
for fill in result.trades:
    sym = fill.order.symbol
    print(
        f"{fill.filled_at:<25} {fill.order.side:>4}  "
        f"{fill.order.shares:>6}  {NAMES.get(sym, sym):<12} "
        f"{fill.filled_price:>10.4f}"
    )

---
## 8. 查看宽表

引擎运行后，每个 symbol 的 DataFrame 都被逐步加宽：

```
原始行情 → +mom +vol +ram → +active
```

In [ ]:
df_nasdaq = result.mktdata["513100.SS"]
print(f"宽表列: {list(df_nasdaq.columns)}")
print()
print("纳指100ETF 最后 5 行：")
print(df_nasdaq[["close", "mom", "vol", "ram", "active"]].tail().to_string(float_format="%.4f"))

---
## 9. 不同 TopNRankingOptimizer 参数对比

TopNRankingOptimizer 的参数选择影响集中度和风险。我们对比不同 n 和 max_weight 的表现：

In [ ]:
# 对比不同 TopNRankingOptimizer 参数
variants = {
    "Top1 (集中)": TopNRankingOptimizer(score_col="ram", n=1, max_weight=1.0),
    "Top2 (均衡)": TopNRankingOptimizer(score_col="ram", n=2, max_weight=0.6),
    "Top3 (分散)": TopNRankingOptimizer(score_col="ram", n=3, max_weight=0.5),
}

results = {}
for label, opt in variants.items():
    sig = Threshold()
    sig.required_indicators = {
        "mom": (Momentum(), {"column": "close", "period": 20}),
        "vol": (RollingVolatility(), {"column": "close", "period": 20}),
        "ram": (Ratio(), {"col_a": "mom", "col_b": "vol"}),
    }
    b = SimBroker()
    r = Engine().run(
        Strategy(
            name=f"rotation_{label}",
            universe=StaticUniverse(("513100.SS", "510300.SS", "518880.SS")),
            signals={
                "active": (sig, {"column": "ram", "threshold": 0, "relationship": "gt"}),
            },
            portfolio=opt,
        ),
        market=LocalMarketDataProvider(),
        broker=b,
        start="2024-11-15", end="2026-02-28",
        initial_cash=100_000.0,
    )
    results[label] = r

# 对比表
header = f"{'':>16}" + "".join(f"{label:>16}" for label in results)
print(header)
print("-" * len(header))
for metric, fn in [
    ("总收益率", lambda r: f"{r.total_return():.2%}"),
    ("年化收益率", lambda r: f"{r.annualized_return():.2%}"),
    ("Sharpe", lambda r: f"{r.sharpe_ratio():.2f}"),
    ("最大回撤", lambda r: f"{r.max_drawdown():.2%}"),
    ("交易次数", lambda r: f"{len(r.trades)}"),
    ("期末资产", lambda r: f"{r.equity_curve[-1][1]:,.0f}"),
]:
    vals = "".join(f"{fn(r):>16}" for r in results.values())
    print(f"{metric:>16}{vals}")

---
## 10. 分阶段执行

轮动策略支持 `run_through` 分阶段执行。

这在调试时非常有用——先验证指标是否合理（通过信号的 required_indicators 自动计算），再看 Signal 权重分配是否符合预期：

In [ ]:
# 只执行到 Signal 阶段 — 观察指标和信号计算结果
b = SimBroker()
result_sig = Engine().run(
    strategy,
    market=LocalMarketDataProvider(),
    broker=b,
    start="2024-11-15", end="2026-02-28",
    run_through="signal",
)

print(f"交易次数: {len(result_sig.trades)}（预期为 0，Signal 阶段不执行交易）")
print()

# 查看某一天三只 ETF 的指标和信号
NAMES = {"513100.SS": "纳指100ETF", "510300.SS": "沪深300ETF", "518880.SS": "黄金ETF"}
for sym, name in NAMES.items():
    df = result_sig.mktdata[sym]
    cols = [c for c in ["close", "mom", "vol", "ram", "active"] if c in df.columns]
    sample = df.loc["2025-06-01":"2025-06-10", cols]
    if not sample.empty:
        print(f"\n{name}:")
        print(sample.to_string(float_format="%.4f"))

---
## 小结

本教程覆盖了轮动策略的完整构建流程：

| 组件 | 职责 | 关键参数 |
|------|------|----------|
| `Momentum` | 计算 N 日动量 | period=20 |
| `RollingVolatility` | 计算 N 日波动率 | period=20 |
| `Ratio` | 两列之比（风险调整动量） | col_a, col_b |
| `Threshold` | per-symbol 布尔信号（携带 required_indicators） | column, threshold |
| `TopNRankingOptimizer` | 截面排名 → 目标权重 | score_col, n, max_weight |

### 架构要点

| | 旧架构 | 新架构 |
|---|---------|----------|
| 指标声明 | Strategy.indicators | Signal.required_indicators |
| 截面排名 | TopNRanking (Signal) | **TopNRankingOptimizer (Portfolio)** |
| 权重分配 | RebalanceRule | PortfolioOptimizer |
| 规则 | Strategy.entry_rules / exit_rules / rebalance_rules | Engine.run(rules=[...]) |
| Strategy 字段 | name, universe, indicators, signals, *_rules | name, universe, signals, portfolio |

### 关键设计原则

- **Signal 只做 per-symbol 计算** — 截面权重分配由 PortfolioOptimizer 处理
- **指标预热期** — Momentum(20) + RollingVolatility(20) 需要 ~21 个交易日
- **权重上限归现金** — 超过 max_weight 的部分不重新分配，自动降低总仓位
- **required_indicators** — 指标通过信号声明依赖，Engine 自动按序计算
- **PortfolioOptimizer** — 策略声明式组合，权重分配逻辑独立可替换